In [4]:
from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_community.document_loaders import PyPDFLoader
load_dotenv()

/var/folders/fq/kr6gv5l17pd4j572n0_n3f2c0000gn/T/ipykernel_3020/607174786.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


True

In [2]:
# pip install langchain_community

In [7]:
# pip install pypdf

In [8]:
loader = PyPDFLoader("bajaj_finance_policy_prose_v1.pdf")
pages = loader.load()

In [ ]:
print(pages[0].metadata)

{'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-05-28T12:50:55+00:00', 'author': 'BajajBot Training Data', 'keywords': '', 'moddate': '2026-05-28T12:50:55+00:00', 'subject': '(unspecified)', 'title': 'Bajaj Finance Helpdesk Knowledge Base — Prose Edition', 'trapped': '/False', 'source': 'bajaj_finance_policy_prose_v1.pdf', 'total_pages': 14, 'page': 0, 'page_label': '1'}


In [17]:
# pages[0].metadata['keywords'] = ['Bajaj Finance', 'Personal Loan', 'Policy']

In [18]:
# pages[0].metadata

In [19]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=350,
                                          separators=["\n\n", "\n", " ", ""],
                                          chunk_overlap=64 # character overlap
                                          )


chunks = splitter.split_documents(pages)

In [23]:
# pip install sentence_transformers

In [24]:
# chunks
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('all-MiniLM-L6-v2')

/Users/rahultiwari/Documents/02_Freelancing/Rbyte_project/Sathish_AI/test-env/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9873.46it/s]


In [28]:
chunk_text = [chunk.page_content for chunk in chunks]
chunk_meta = [chunk.metadata for chunk in chunks]

In [30]:
chunk_vectors = model.encode(
    chunk_text,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True # makes cosin --> faster
)

Batches: 100%|██████████| 5/5 [00:01<00:00,  3.29it/s]


In [33]:
len(chunk_vectors[0])

384

In [ ]:
# chunk_vectors[0]

array([ 3.51866824e-03,  4.00002673e-02, -8.39214623e-02, -5.15348138e-03,
       -8.08845926e-03,  4.22597900e-02,  3.65482876e-03,  7.97159597e-02,
       -5.43158837e-02, -4.02022377e-02,  2.51007043e-02, -2.07973309e-02,
        4.60176878e-02,  1.32275640e-03,  2.88321264e-02,  5.18208556e-03,
        3.02016102e-02, -2.53717396e-02, -1.33666804e-03, -1.10513149e-02,
        4.56920825e-02,  3.84921371e-03, -1.11316144e-02, -2.88040992e-02,
       -3.42000164e-02, -1.12694629e-01,  3.10664177e-02, -3.24472301e-02,
       -5.43849505e-02, -4.06140164e-02, -1.03370445e-02,  1.01668708e-01,
        2.94144284e-02,  3.17434818e-02,  6.38263077e-02,  3.17767709e-02,
       -3.24789584e-02,  5.44509664e-02,  2.54387520e-02,  3.79820131e-02,
       -8.78702626e-02, -1.97591707e-02,  7.83351934e-05, -3.70970927e-02,
        2.25079264e-02, -4.99753207e-02, -2.32824665e-02, -3.25548113e-03,
        5.69729246e-02,  4.02206033e-02, -1.15810424e-01, -1.75422840e-02,
        1.47290463e-02,  

In [35]:
store = {
    'vectors':chunk_vectors,
    'text':chunk_text,
    'meta':chunk_meta,
    'model':'all-MiniLM-L6-v2'
}

import pickle

with open("bfl_store.pkl", "wb") as f:
    pickle.dump(store, f)